# Lab 1: Artificial Neural Networks in Keras

**DATA425 | Foundations of Deep Learning**

## What this lab is about

In this lab, we turn the first neural network ideas from the lectures into working code. The main goal is not to build the biggest model. The goal is to understand what each part of a small neural network is doing.

By the end, you should be able to explain the difference between:

- a linear classifier and a neural network with hidden layers;
- a sigmoid output for binary classification and a softmax output for multi-class classification;
- a model that can draw only a straight decision boundary and a model that can learn a curved one.

We will use simple two-dimensional datasets because they are easy to plot. This lets us see the decision boundary directly, which is much harder with real image or text data. The same ideas carry over to larger datasets later in the course.


## 0. Setup

Run the setup cells first. They import the libraries we use, install missing packages when possible, and set random seeds so that repeated runs give similar results.

A seed does not make training perfectly identical on every machine, but it makes the notebook much easier to discuss because your plots and numbers should be close to the examples shown here.

We use:

- **NumPy** for arrays and numerical work;
- **Matplotlib** for plots;
- **scikit-learn** for small toy datasets and baseline models;
- **TensorFlow/Keras** for building neural networks.


In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED = {
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",
    "tensorflow": "tensorflow"
}

missing = [package for module, package in REQUIRED.items()
           if importlib.util.find_spec(module) is None]

if missing:
    print("Installing missing packages:", ", ".join(missing))
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            "Automatic installation failed. Install these packages manually or run this notebook "
            "in an environment with internet access: " + ", ".join(missing)
        ) from exc
else:
    print("All required packages are already installed.")

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ.setdefault("TF_ENABLE_ONEDNN_OPTS", "0")

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.datasets import make_classification, make_moons, make_circles
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

try:
    tf.config.threading.set_inter_op_parallelism_threads(1)
    tf.config.threading.set_intra_op_parallelism_threads(1)
except RuntimeError:
    # TensorFlow may already be initialised in an interactive notebook.
    pass

SEED = 425
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["font.size"] = 12
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

print(f"TensorFlow version: {tf.__version__}")
print("Setup complete.")

After the setup finishes, the notebook has everything it needs. If the install cell fails, check that you are using an environment with internet access or install the listed packages manually before continuing.


## 1. Helper functions

The next cell defines plotting and evaluation helpers. These are here to keep the rest of the notebook focused on the modelling ideas rather than on repeated plotting code.

You do not need to memorise these helper functions. Read their names and notice what they are for:

- `plot_points` shows a labelled dataset;
- `plot_binary_boundary` colours the input space by predicted probability and draws the 0.5 decision boundary;
- `plot_multiclass_boundary` shows which class the model predicts in each region;
- `plot_history` shows training and validation curves;
- `evaluate_binary` prints a compact accuracy summary.

When you work with more complex datasets, the same pattern is useful: write small helper functions once, then use them repeatedly to compare models clearly.


In [ ]:
def plot_points(X, y, title="Dataset", ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="RdYlBu", s=25,
               alpha=0.85, edgecolors="k", linewidth=0.3)
    ax.set_xlabel("feature 1")
    ax.set_ylabel("feature 2")
    ax.set_title(title)
    return ax


def _probability_vector(predict_fn, grid):
    values = np.asarray(predict_fn(grid))
    if values.ndim == 2 and values.shape[1] > 1:
        values = values[:, 1]
    return values.reshape(-1)


def plot_binary_boundary(predict_fn, X, y, title="Decision boundary", ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 5))
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 60), np.linspace(y_min, y_max, 60))
    grid = np.c_[xx.ravel(), yy.ravel()]
    probs = _probability_vector(predict_fn, grid).reshape(xx.shape)
    ax.contourf(xx, yy, probs, levels=30, cmap="RdYlBu", alpha=0.65, vmin=0, vmax=1)
    ax.contour(xx, yy, probs, levels=[0.5], colors="black", linewidths=2)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="RdYlBu", s=20, edgecolors="k", linewidth=0.3)
    ax.set_xlabel("feature 1")
    ax.set_ylabel("feature 2")
    ax.set_title(title)
    return ax


def plot_multiclass_boundary(predict_fn, X, y, title="Decision regions", ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 5))
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 60), np.linspace(y_min, y_max, 60))
    grid = np.c_[xx.ravel(), yy.ravel()]
    probs = np.asarray(predict_fn(grid))
    labels = np.argmax(probs, axis=1).reshape(xx.shape)
    ax.contourf(xx, yy, labels, levels=np.arange(probs.shape[1] + 1) - 0.5,
                cmap="Set3", alpha=0.75)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="Set1", s=22, edgecolors="k", linewidth=0.3)
    ax.set_xlabel("feature 1")
    ax.set_ylabel("feature 2")
    ax.set_title(title)
    return ax


def plot_history(history, title="Training history"):
    hist = history.history
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(hist["loss"], label="training loss")
    if "val_loss" in hist:
        ax.plot(hist["val_loss"], label="validation loss")
    ax.set_xlabel("epoch")
    ax.set_ylabel("loss")
    ax.set_title(title)
    ax.legend()
    plt.show()


def evaluate_binary(predict_fn, X, y, label="model"):
    probs = _probability_vector(predict_fn, X)
    preds = (probs >= 0.5).astype(int)
    print(f"{label} accuracy: {accuracy_score(y, preds):.3f}")
    print("Confusion matrix:")
    print(confusion_matrix(y, preds))


def predict_keras(model, X):
    return model.predict(X, verbose=0)

The most important helper is the decision boundary plot. It answers the visual question: **what has the model learned to separate?** If the boundary has the wrong shape, more training epochs alone may not fix the problem. The architecture may be too simple.


## 2. A straight-line problem

We begin with the friendliest kind of classification problem. Each point has two input features, and the target is either class 0 or class 1.

The classes are almost separable by a straight line. This matters because a simple perceptron or logistic regression model is a linear classifier. It can learn a straight decision boundary, but it cannot naturally bend that boundary around complicated shapes.

We also split the data into training and test sets. The model is fitted on the training set. The test set is kept aside so we can check whether the model generalises to points it did not train on.


In [ ]:
X_linear, y_linear = make_classification(
    n_samples=600,
    n_features=2,
    n_redundant=0,
    n_informative=2,
    n_clusters_per_class=1,
    class_sep=1.8,
    random_state=SEED,
)

X_train, X_test, y_train, y_test = train_test_split(
    X_linear, y_linear, test_size=0.25, random_state=SEED, stratify=y_linear
)

plot_points(X_linear, y_linear, "Linearly separable data")
plt.show()

Look at the plot before you move on. If you can imagine drawing one straight line that separates the colours, then a linear model is a sensible first choice.


### Logistic regression as a reference model

Logistic regression is a useful baseline for binary classification. It learns a linear score, then turns that score into a probability with a sigmoid function.

A baseline is not something we use only because it is simple. A baseline gives us a reference point. If a more complicated neural network cannot beat a simple baseline, then the extra complexity may not be helping.

In the plot, the black contour is where the model predicts probability 0.5 for class 1. Points on one side are classified as class 0, and points on the other side are classified as class 1.


In [ ]:
lr_model = LogisticRegression(random_state=SEED)
lr_model.fit(X_train, y_train)

def lr_predict(grid):
    return lr_model.predict_proba(grid)[:, 1]

print("Logistic regression test accuracy:", f"{lr_model.score(X_test, y_test):.3f}")
plot_binary_boundary(lr_predict, X_linear, y_linear, "Logistic regression learns a straight boundary")
plt.show()

The boundary should be close to a straight line. That is exactly what we expect from a linear classifier. The model is not failing to be creative. Its architecture only allows a straight split in the original feature space.


### The same model in Keras

Now we build the same basic idea using Keras. The model has one dense output unit with a sigmoid activation:

$$\hat y = \sigma(w_1x_1 + w_2x_2 + b).$$

This output can be read as:

$$P(\text{class}=1 \mid x_1, x_2).$$

We compile the model with:

- `binary_crossentropy`, because the target has two classes;
- `Adam`, because it is a practical optimiser that usually works well without too much tuning;
- `accuracy`, because it is easy to interpret for this balanced toy problem.

The key idea is that a one-layer sigmoid network is not yet a deep network. It is a linear classifier written in neural network language.


In [ ]:
tf.keras.utils.set_random_seed(SEED)

keras_linear = keras.Sequential([
    keras.Input(shape=(2,)),
    layers.Dense(1, activation="sigmoid"),
])
keras_linear.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.03),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
history_linear = keras_linear.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=5,
    batch_size=32,
    verbose=0,
)

plot_history(history_linear, "One-layer Keras classifier")
evaluate_binary(lambda z: predict_keras(keras_linear, z), X_test, y_test, "Keras one-layer model")
plot_binary_boundary(lambda z: predict_keras(keras_linear, z), X_linear, y_linear,
                     "One-layer Keras model learns a straight boundary")
plt.show()

The Keras model should learn a similar straight boundary to logistic regression. This is a useful checkpoint: changing the library does not change what the architecture can represent.


## 3. When a straight line is not enough

The previous dataset was designed to be easy for a linear classifier. Real data is often not that convenient.

The moons dataset has two classes that curve around each other. A straight decision boundary will cut through the shapes and make systematic mistakes. This is where hidden layers become useful.

We also standardise the input features before fitting the neural networks. Standardisation puts features on a similar scale, which usually makes optimisation smoother.


In [ ]:
X_moons, y_moons = make_moons(n_samples=600, noise=0.20, random_state=SEED)
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_moons, y_moons, test_size=0.25, random_state=SEED, stratify=y_moons
)

scaler_moons = StandardScaler().fit(X_train_m)
X_train_m_s = scaler_moons.transform(X_train_m)
X_test_m_s = scaler_moons.transform(X_test_m)

plot_points(X_moons, y_moons, "Moons data: a curved decision boundary is needed")
plt.show()

The plot should make the problem clear: if you try to separate the classes with one straight line, there is no good place to put it. We need a model that can transform the inputs before classifying them.


### A linear neural network struggles

First we try the same one-layer sigmoid model on the moons data. This is not a bad model because the code is wrong. It is a bad model because the **representation is too limited** for the shape of the data.

This is an important modelling habit: when performance is poor, ask whether the model is undertrained, whether the optimisation is difficult, or whether the architecture cannot express the pattern you need.


In [ ]:
tf.keras.utils.set_random_seed(SEED)

moons_linear = keras.Sequential([
    keras.Input(shape=(2,)),
    layers.Dense(1, activation="sigmoid"),
])
moons_linear.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.03),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
moons_linear.fit(
    X_train_m_s, y_train_m,
    validation_data=(X_test_m_s, y_test_m),
    epochs=5,
    batch_size=32,
    verbose=0,
)

def moons_linear_predict(grid):
    return predict_keras(moons_linear, scaler_moons.transform(grid))

plot_binary_boundary(moons_linear_predict, X_moons, y_moons, "Linear model cannot bend enough")
plt.show()
evaluate_binary(lambda z: predict_keras(moons_linear, z), X_test_m_s, y_test_m, "Linear model on moons")

The boundary should still be straight. It may move to the best straight split it can find, but it cannot bend around the moons. This is a clear example of underfitting caused by an overly simple model class.


### Hidden layers can bend the decision boundary

We now add hidden layers with ReLU activations. A hidden layer learns new features from the original inputs. The output layer then uses those learned features to classify the point.

A useful way to think about this is:

1. The first hidden layer creates several simple transformations of the input.
2. The second hidden layer combines those transformations.
3. The final sigmoid unit turns the result into a probability.

ReLU is important because it introduces nonlinearity. Without nonlinear activations, stacking dense layers would still collapse into one linear transformation.


In [ ]:
tf.keras.utils.set_random_seed(SEED)

moons_deep = keras.Sequential([
    keras.Input(shape=(2,)),
    layers.Dense(16, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])
moons_deep.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
history_moons_deep = moons_deep.fit(
    X_train_m_s, y_train_m,
    validation_data=(X_test_m_s, y_test_m),
    epochs=8,
    batch_size=32,
    verbose=0,
)

def moons_deep_predict(grid):
    return predict_keras(moons_deep, scaler_moons.transform(grid))

plot_history(history_moons_deep, "Hidden layers on moons")
plot_binary_boundary(moons_deep_predict, X_moons, y_moons, "Hidden layers learn a curved boundary")
plt.show()
evaluate_binary(lambda z: predict_keras(moons_deep, z), X_test_m_s, y_test_m, "Deep model on moons")

The boundary should now be curved. This does not mean hidden layers magically solve every problem. It means they give the model a richer set of shapes it can learn.


## 4. Multi-class classification with softmax

Binary classification uses one sigmoid output because there is one probability to estimate: class 1 versus not class 1.

Multi-class classification needs one output unit per class. The softmax function turns the output scores into probabilities that add to 1. For three classes, the model returns something like:

$$[P(y=0), P(y=1), P(y=2)].$$

We use `sparse_categorical_crossentropy` because the labels are stored as class numbers such as 0, 1, and 2 rather than as one-hot encoded vectors.

The spiral dataset is deliberately harder than the first dataset. It helps us see the difference between a linear softmax classifier and a softmax classifier with hidden layers.


In [ ]:
def make_spiral(n_per_class=180, classes=3, noise=0.18, seed=SEED):
    rng = np.random.default_rng(seed)
    X_parts, y_parts = [], []
    for class_id in range(classes):
        radius = np.linspace(0.1, 1.0, n_per_class)
        theta = np.linspace(class_id * 4, (class_id + 1) * 4, n_per_class)
        theta = theta + rng.normal(0, noise, n_per_class)
        X_parts.append(np.c_[radius * np.sin(theta), radius * np.cos(theta)])
        y_parts.append(np.full(n_per_class, class_id))
    return np.vstack(X_parts).astype("float32"), np.concatenate(y_parts)

X_spiral, y_spiral = make_spiral()
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_spiral, y_spiral, test_size=0.25, random_state=SEED, stratify=y_spiral
)
scaler_spiral = StandardScaler().fit(X_train_s)
X_train_s_scaled = scaler_spiral.transform(X_train_s)
X_test_s_scaled = scaler_spiral.transform(X_test_s)

plot_points(X_spiral, y_spiral, "Three-class spiral data")
plt.show()

The three colours twist around each other. A model that only creates straight regions will struggle. A neural network with hidden layers has a better chance because it can transform the feature space before applying softmax.


In [ ]:
tf.keras.utils.set_random_seed(SEED)

softmax_model = keras.Sequential([
    keras.Input(shape=(2,)),
    layers.Dense(3, activation="softmax"),
])
softmax_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.03),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
softmax_model.fit(
    X_train_s_scaled, y_train_s,
    validation_data=(X_test_s_scaled, y_test_s),
    epochs=8,
    batch_size=32,
    verbose=0,
)

spiral_deep = keras.Sequential([
    keras.Input(shape=(2,)),
    layers.Dense(32, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(3, activation="softmax"),
])
spiral_deep.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
history_spiral = spiral_deep.fit(
    X_train_s_scaled, y_train_s,
    validation_data=(X_test_s_scaled, y_test_s),
    epochs=8,
    batch_size=32,
    verbose=0,
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_multiclass_boundary(
    lambda z: predict_keras(softmax_model, scaler_spiral.transform(z)),
    X_spiral, y_spiral, "Softmax only: linear regions", ax=axes[0]
)
plot_multiclass_boundary(
    lambda z: predict_keras(spiral_deep, scaler_spiral.transform(z)),
    X_spiral, y_spiral, "Hidden layers: curved regions", ax=axes[1]
)
plt.tight_layout()
plt.show()

plot_history(history_spiral, "Deep softmax classifier")
probs = predict_keras(spiral_deep, X_test_s_scaled)
preds = np.argmax(probs, axis=1)
print("Deep softmax test accuracy:", f"{accuracy_score(y_test_s, preds):.3f}")
print(classification_report(y_test_s, preds))

Compare the two decision region plots carefully. The softmax-only model creates simple linear regions. The model with hidden layers can create curved regions that follow the spiral structure more closely.


## Summary

You have now built the first core pieces of a neural network workflow:

- A single sigmoid unit behaves like logistic regression for binary classification.
- Linear models are useful baselines, but they can only draw linear decision boundaries in the original feature space.
- Hidden layers plus nonlinear activations allow a network to learn more flexible boundaries.
- Softmax is the standard output activation for multi-class classification.
- Training and test splits help us check whether a model generalises beyond the data it fitted.

The main takeaway is that architecture controls what kinds of patterns a model can represent. Optimisation teaches the model where to place its boundary, but the architecture determines what shapes are possible in the first place.
